# **kwargs, Decorators & Generators

In [ ]:
# Day 22 — **kwargs, Decorators & Generators
# Three powerful Python features that take your skills to the next level
#
# Today we'll cover:
#   1. **kwargs — keyword arguments (like *args but with names)
#   2. Combining *args and **kwargs
#   3. Decorators — wrapping functions with extra behaviour
#   4. Built-in decorators: @staticmethod, @classmethod, @property
#   5. Generators — lazy sequences with yield
#
# These are INTERMEDIATE concepts, but essential because:
# - **kwargs is used in every framework (Flask, Django, MLflow)
# - Decorators are used in Flask routes, FastAPI endpoints, pytest
# - Generators are used for processing large data efficiently

## **kwargs — Keyword Arguments

In [ ]:
# In Day 7, we learned *args — accepts any number of POSITIONAL arguments
# **kwargs accepts any number of KEYWORD (named) arguments
#
# *args  -> collected into a TUPLE
# **kwargs -> collected into a DICTIONARY

# Recap of *args:
def add(*args):
    return sum(args)

print(add(1, 2, 3))           # 6  -> args = (1, 2, 3)
print(add(10, 20, 30, 40))    # 100 -> args = (10, 20, 30, 40)

In [ ]:
# **kwargs example:
# Think of kwargs as 'keyword arguments'
# They are passed as key=value pairs and received as a dictionary

def print_info(**kwargs):
    print(f'kwargs type: {type(kwargs)}')     # <class 'dict'>
    for key, value in kwargs.items():
        print(f'  {key}: {value}')

print_info(name='Krishna', age=25, city='Chennai')
# kwargs type: <class 'dict'>
#   name: Krishna
#   age: 25
#   city: Chennai

print()
print_info(host='10.0.0.1', port=8080, protocol='https')
# kwargs type: <class 'dict'>
#   host: 10.0.0.1
#   port: 8080
#   protocol: https

# KEY POINT: **kwargs lets you pass ANY key-value pairs
# You don't need to define the parameter names in advance
# The caller decides what to pass!

In [ ]:
# Real-world use case: Config builder

def build_config(app_name, **kwargs):
    config = {'app_name': app_name}
    config.update(kwargs)          # merge all keyword args into the config
    return config

# Simple config
cfg1 = build_config('api-gateway', port=8080)
print(cfg1)    # {'app_name': 'api-gateway', 'port': 8080}

# Detailed config
cfg2 = build_config('payment-service',
                     port=8443,
                     replicas=3,
                     env='production',
                     db_host='postgres-svc')
print(cfg2)
# {'app_name': 'payment-service', 'port': 8443, 'replicas': 3,
#  'env': 'production', 'db_host': 'postgres-svc'}

# This pattern is used EVERYWHERE:
# Flask: app.run(host='0.0.0.0', port=5000, debug=True)
# MLflow: mlflow.log_params(lr=0.01, epochs=10, batch_size=32)

### Combining *args and **kwargs

In [ ]:
# You can use BOTH *args and **kwargs in the same function
# Order matters: regular params -> *args -> **kwargs

def super_function(required_param, *args, **kwargs):
    print(f'Required: {required_param}')
    print(f'Extra positional (args): {args}')
    print(f'Extra keyword (kwargs): {kwargs}')

super_function('hello', 1, 2, 3, name='Krishna', age=25)
# Required: hello
# Extra positional (args): (1, 2, 3)
# Extra keyword (kwargs): {'name': 'Krishna', 'age': 25}

# This is how many frameworks accept flexible inputs
# Example in pandas: pd.DataFrame(data, index=..., columns=..., dtype=...)

In [ ]:
# Unpacking dicts with ** (the reverse operation)

# You can UNPACK a dictionary into keyword arguments using **

def deploy(app_name, version, replicas):
    print(f'Deploying {app_name} {version} with {replicas} replicas')

# Instead of calling:
deploy('api-gateway', 'v2.0', 3)

# You can pass a dict:
config = {'app_name': 'api-gateway', 'version': 'v2.0', 'replicas': 3}
deploy(**config)     # ** unpacks the dict into keyword arguments

# Both calls produce: Deploying api-gateway v2.0 with 3 replicas

# This is very useful when you read config from a JSON file
# and want to pass it directly to a function!

## Decorators

In [ ]:
# A decorator is a function that WRAPS another function
# It adds extra behaviour BEFORE and/or AFTER the original function
# WITHOUT modifying the original function's code
#
# Real-world analogy:
# Think of a decorator like gift wrapping 🎁
# The gift (your function) stays the same
# The wrapping (decorator) adds something extra on the outside
# (like a bow, a tag, or a card)
#
# Where you've already seen decorators:
#   @abstractmethod  (Day 11 — OOP Abstraction)
#   @app.route('/')  (Flask web framework)
#   @pytest.fixture  (pytest testing)
#   @staticmethod    (OOP)

# Let's build a decorator from scratch to understand how it works

In [ ]:
# Step 1: Understanding that functions can be passed around

# In Python, functions are OBJECTS — you can pass them as arguments!

def greet():
    return 'Hello!'

# Assign the function to another variable (no parentheses = no calling)
say_hello = greet       # say_hello now POINTS to the same function
print(say_hello())      # Hello!

# Pass a function as an argument
def call_function(func):
    print('About to call the function...')
    result = func()     # call whatever function was passed
    print(f'Function returned: {result}')

call_function(greet)    # passing greet (not greet()) — the function itself
# About to call the function...
# Function returned: Hello!

In [ ]:
# Step 2: Creating a simple decorator

# A decorator is a function that:
# 1. Takes a function as input
# 2. Defines a WRAPPER function inside
# 3. The wrapper adds extra behaviour and calls the original function
# 4. Returns the wrapper

def logger(func):
    def wrapper(*args, **kwargs):    # wrapper accepts any arguments
        print(f'[LOG] Calling {func.__name__}...')
        result = func(*args, **kwargs)    # call the original function
        print(f'[LOG] {func.__name__} finished.')
        return result
    return wrapper

# Using the decorator with @
@logger
def add(a, b):
    return a + b

@logger
def greet(name):
    return f'Hello, {name}!'

result = add(3, 5)
# [LOG] Calling add...
# [LOG] add finished.
print(f'Result: {result}')    # Result: 8

print()

message = greet('Krishna')
# [LOG] Calling greet...
# [LOG] greet finished.
print(message)    # Hello, Krishna!

In [ ]:
# What does @logger actually do?

# @logger
# def add(a, b):
#     return a + b
#
# is EXACTLY the same as:
#
# def add(a, b):
#     return a + b
# add = logger(add)     <- replaces add with the wrapped version

# The @ syntax is just a shortcut (syntactic sugar)
# It makes the code cleaner and more readable

In [ ]:
# Practical decorator: Timer (measure how long a function takes)

import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()                     # record start time
        result = func(*args, **kwargs)          # run the original function
        end = time.time()                       # record end time
        duration = end - start
        print(f'[TIMER] {func.__name__} took {duration:.4f} seconds')
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)          # simulate a slow operation (1 second)
    return 'Done!'

result = slow_function()
# [TIMER] slow_function took 1.0012 seconds
print(result)    # Done!

# You can add @timer to ANY function to measure its performance
# Without changing the function's code at all!

### Built-in Decorators: @staticmethod, @classmethod, @property

In [ ]:
# @staticmethod — a method that doesn't access self or cls
# It's basically a regular function that lives inside a class

class MathHelper:
    @staticmethod
    def add(a, b):         # no 'self' parameter!
        return a + b

    @staticmethod
    def is_even(n):
        return n % 2 == 0

# You can call it WITHOUT creating an object
print(MathHelper.add(3, 5))      # 8
print(MathHelper.is_even(10))    # True

# When to use:
# When the method doesn't need access to instance data (self)
# but logically belongs to the class

In [ ]:
# @classmethod — a method that receives the CLASS itself (not an instance)
# First parameter is 'cls' (the class), not 'self' (an instance)

class Server:
    total_count = 0     # class-level attribute (shared by all instances)

    def __init__(self, name, ip):
        self.name = name
        self.ip = ip
        Server.total_count += 1

    @classmethod
    def get_count(cls):      # 'cls' is the class itself (Server)
        return cls.total_count

    @classmethod
    def from_string(cls, server_string):    # factory method
        name, ip = server_string.split(':')
        return cls(name.strip(), ip.strip())    # creates a new Server

s1 = Server('web-01', '10.0.0.1')
s2 = Server('db-01', '10.0.0.2')
s3 = Server.from_string('cache-01 : 10.0.0.3')    # using the factory method

print(Server.get_count())     # 3
print(s3.name, s3.ip)         # cache-01 10.0.0.3

In [ ]:
# @property — makes a method LOOK like an attribute
# Lets you access it without parentheses: obj.area instead of obj.area()

class Circle:
    def __init__(self, radius):
        self._radius = radius     # using _ to suggest it's internal

    @property
    def radius(self):             # getter — read the value
        return self._radius

    @radius.setter
    def radius(self, value):      # setter — set the value with validation
        if value < 0:
            raise ValueError('Radius cannot be negative!')
        self._radius = value

    @property
    def area(self):               # computed property — calculated on the fly
        return 3.14159 * self._radius ** 2

c = Circle(5)
print(c.radius)       # 5       -> looks like an attribute, but it's a method!
print(c.area)         # 78.54   -> calculated automatically

c.radius = 10         # uses the setter (with validation)
print(c.area)         # 314.16  -> automatically recalculated

# c.radius = -3       # ValueError: Radius cannot be negative!

# @property gives you CLEAN syntax (no parentheses) with VALIDATION behind the scenes

## Generators

In [ ]:
# A generator is a function that produces values ONE AT A TIME
# Instead of creating and returning an entire list at once,
# it YIELDS one value, pauses, then yields the next when asked
#
# Real-world analogy:
# Regular function = a factory that builds ALL products, puts them in a
#                     warehouse, then ships the entire warehouse to you
# Generator = a factory that builds ONE product, ships it to you,
#             then waits for you to ask for the next one
#
# Why generators?
# - MEMORY EFFICIENT — doesn't store everything in memory at once
# - Great for processing LARGE files, INFINITE sequences, or STREAMS
# - The keyword is 'yield' (instead of 'return')

# Regular function (stores EVERYTHING in memory)
def get_squares_list(n):
    result = []
    for i in range(n):
        result.append(i ** 2)
    return result

# Generator function (produces one at a time)
def get_squares_gen(n):
    for i in range(n):
        yield i ** 2         # 'yield' instead of 'append' + 'return'

# Using the regular function (creates a full list in memory)
squares_list = get_squares_list(5)
print(type(squares_list))    # <class 'list'>
print(squares_list)          # [0, 1, 4, 9, 16]

# Using the generator (creates a generator object)
squares_gen = get_squares_gen(5)
print(type(squares_gen))     # <class 'generator'>
print(squares_gen)           # <generator object get_squares_gen at 0x...>

In [ ]:
# How to GET values from a generator:

# Method 1: next() — get one value at a time
gen = get_squares_gen(5)
print(next(gen))    # 0   -> first value
print(next(gen))    # 1   -> second value
print(next(gen))    # 4   -> third value
# ... and so on. When no more values: raises StopIteration

# Method 2: for loop — most common way
print('\nAll squares:')
for square in get_squares_gen(5):
    print(square, end=' ')
print()    # 0 1 4 9 16

# Method 3: Convert to list (defeats the purpose of memory saving, but useful sometimes)
all_squares = list(get_squares_gen(5))
print(all_squares)    # [0, 1, 4, 9, 16]

In [ ]:
# When generators REALLY shine: LARGE data

# Imagine you need to process a 10GB log file
# Regular approach: read ALL lines into a list -> out of memory!
# Generator approach: read ONE line at a time -> works fine!

def read_large_file(filepath):
    """Generator that reads a file line by line."""
    with open(filepath, 'r') as f:
        for line in f:
            yield line.strip()

# This reads ONE line at a time, no matter how big the file is!
# for line in read_large_file('huge_logfile.log'):
#     if '[ERROR]' in line:
#         print(line)

# The file is never fully loaded into memory

In [ ]:
# Generator Expressions — like list comprehensions but lazy

# List comprehension uses [] -> creates the entire list in memory
squares_list = [x ** 2 for x in range(1000000)]    # 1 million items in memory!

# Generator expression uses () -> creates values one at a time
squares_gen = (x ** 2 for x in range(1000000))     # almost no memory used!

print(type(squares_list))    # <class 'list'>
print(type(squares_gen))     # <class 'generator'>

# You can still loop through the generator:
# for sq in squares_gen:
#     do_something(sq)

# Or use it with functions that accept iterables:
total = sum(x ** 2 for x in range(1000000))    # sum of first million squares
print(f'Sum: {total}')    # calculated without storing all values!

In [ ]:
# Summary: yield vs return
#
# return:
#   - Sends a value back and TERMINATES the function
#   - All local variables are LOST
#   - Next call starts from the BEGINNING
#
# yield:
#   - Sends a value back and PAUSES the function
#   - All local variables are PRESERVED (remembered)
#   - Next call RESUMES from where it paused
#
# Use generators when:
# 1. Processing large datasets (files, API responses)
# 2. Creating infinite sequences (counter, fibonacci)
# 3. You only need values one at a time (streaming)
# 4. Memory efficiency is important

In [ ]:
# Practice Questions:

# 1. Write a function print_server_config(**kwargs) that accepts
#    any server configuration options and prints them formatted
#    print_server_config(host='web-01', port=8080, env='prod')

# 2. Write a decorator @repeat(n) that makes a function run n times
#    @repeat(3)
#    def say_hello():
#        print('Hello!')
#    say_hello()  # prints 'Hello!' three times
#    Hint: this is a decorator WITH arguments (decorator factory)

# 3. Write a generator function fibonacci(n) that yields the first
#    n Fibonacci numbers: 0, 1, 1, 2, 3, 5, 8, 13...

# 4. Create a class Temperature with:
#    - Private attribute _celsius
#    - @property celsius (getter and setter)
#    - @property fahrenheit (computed: celsius * 9/5 + 32)
#    - Setter should reject temperatures below -273.15 (absolute zero)

# 5. Write a generator expression that produces all even numbers
#    from 1 to 1000. Use sum() to find their total.

### DevOps / AI-MLOps Practice

In [ ]:
# These patterns appear constantly in professional Python code

# Exercise 1: Logging Decorator
# Write a decorator @log_call that:
#   - Prints the function name and arguments when called
#   - Prints the return value when done
#   - Prints the execution time
# Test it on a function deploy(app_name, version, replicas=1)
# Expected output:
#   [LOG] deploy called with args=('api-gateway', 'v2.0') kwargs={'replicas': 3}
#   [LOG] deploy returned: 'Deployed successfully'
#   [LOG] deploy took 0.0012 seconds

# Exercise 2: Retry Decorator
# Write a decorator @retry(max_attempts=3) that:
#   - If the function raises an exception, retries up to max_attempts times
#   - Prints which attempt number it's on
#   - If all attempts fail, raises the last exception
# Test it on a function that randomly fails:
#   @retry(max_attempts=3)
#   def call_api():
#       if random.random() < 0.7: raise ConnectionError('Timeout')
#       return 'Success'

# Exercise 3: Config Builder with **kwargs
# Write a function create_k8s_deployment(name, image, **kwargs):
#   - Required: name, image
#   - Optional kwargs: replicas (default 1), namespace (default 'default'),
#     port, memory_limit, cpu_limit, env_vars (dict)
#   - Returns a config dict with all values
#   - Print the config as formatted JSON

# Exercise 4: Lazy Log Reader (Generator)
# Write a generator function parse_logs(filepath) that:
#   - Reads a log file line by line (memory efficient)
#   - Parses each line into a dict: {'level': 'ERROR', 'timestamp': ..., 'message': ...}
#   - Yields one parsed dict at a time
# Then use it to:
#   - Count errors without loading the entire file
#   - Find the first ERROR and stop (break out of the loop)
#   - Get all ERROR messages into a list

# Why these patterns matter:
# - Flask/FastAPI use decorators for EVERY route (@app.get('/'))
# - MLflow uses **kwargs for logging params (mlflow.log_params(lr=0.01))
# - Generators are used in data pipelines for streaming large datasets
# - @property is used in Kubernetes Python client for resource objects